# Extract Qwen2.5-VL-7B vision-tower features + small-sample probe

**Repo**: domain-bounded-cslr · **Purpose**: run the §22/§23 `0.69 macro-AUC` ceiling test with the next size-up tower (Qwen2.5-VL-7B vs the 3B already tried) on a Kaggle T4 (16 GB).

**Two-phase flow** (controlled by `LIMIT`):
- **Small trial** `LIMIT=40`: extract 40 train + 40 dev features, run a linear OneVsRest probe (train→dev). If `macro_auc` clears ~0.75 we have a green light to spend GPU on the full extract.
- **Full run** `LIMIT=None`: extract all train(4973)+dev(515), optional probe, then zip features for download back to the host.

**The frozen test split is never read** — the uploaded package contains only train/validation records.

In [ ]:
!pip -q install transformers==4.51.3 torch opencv-python-headless numpy scikit-learn
import os, gc, json, zipfile, time, csv
from pathlib import Path
import numpy as np
import torch
import cv2

In [ ]:
# ---------- settings ----------
FRAMES   = 48
IMAGE    = 224
MODEL    = "Qwen/Qwen2.5-VL-7B-Instruct"   # HF id; or a ModelScope path
LIMIT    = 40                                # 40 = small trial; None = full run
RUN_PROBE= True
OUT      = Path("/kaggle/working/ce-csl-qwenvl7b")
device   = "cuda" if torch.cuda.is_available() else "cpu"
print("device", device, "gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
# locate the uploaded package (its root holds manifest.csv)
cands = sorted(Path("/kaggle/input").resolve().glob("*/manifest.csv"))
if not cands:
    raise SystemExit("no manifest.csv under /kaggle/input - upload the pack from pack_kaggle_data.py")
DATA = cands[0].parent
print("dataset root:", DATA)

In [ ]:
# sample_frames: sequential streaming grab, one pass - same [T,224,224,3] contract as clip_features
def sample_frames(video: Path, count: int, size: int = 224):
    if count < 1:
        raise ValueError("count must be >= 1")
    if not video.exists():
        raise FileNotFoundError(video)
    cap = cv2.VideoCapture(str(video))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    if total == 0:
        cap.release()
        raise RuntimeError(f"could not read frame count: {video}")
    wanted = set(np.linspace(0, total - 1, count).round().astype(int).tolist())
    frames = []
    pos = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if pos in wanted:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(cv2.resize(rgb, (size, size)))
        pos += 1
    cap.release()
    while len(frames) < count and frames:      # guard against short files
        frames.append(frames[-1])
    if len(frames) < count:
        raise RuntimeError(f"too few decoded frames ({len(frames)}/{count}): {video}")
    return frames

In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

class FrozenQwenVisionEncoder:
    """Frozen Qwen2.5-VL vision tower -> one vector per frame [T,D] float32."""
    def __init__(self, model_path: str, device: str):
        self.device = torch.device(device)
        self.processor = AutoProcessor.from_pretrained(model_path)
        full = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_path, torch_dtype=torch.bfloat16)
        try:
            self.visual = full.model.visual.to(self.device).eval()   # transformers 4.51
        except AttributeError:
            self.visual = full.visual.to(self.device).eval()
        del full; gc.collect(); torch.cuda.empty_cache()
        self.feature_size = int(self._pool_per_frame(self._forward_zeros(1), 1).shape[-1])
        print("feature_size", self.feature_size)

    def _forward_zeros(self, n):
        probe = np.zeros((224, 224, 3), dtype=np.uint8)
        with torch.no_grad():
            return self.visual(*self._inputs([probe] * n))

    def _inputs(self, frames):
        proc = self.processor.image_processor(images=frames, return_tensors="pt")
        return proc["pixel_values"].to(self.device), proc["image_grid_thw"].to(self.device)

    def _pool_per_frame(self, out, frame_count):
        pooled = getattr(out, "pooler_output", None)
        if pooled is None:
            raise RuntimeError("vision tower returned no pooler_output")
        rows = pooled.shape[0]
        if rows % frame_count:
            raise RuntimeError(f"{rows} tokens not divisible by {frame_count}")
        per = rows // frame_count
        return pooled.reshape(frame_count, per, pooled.shape[-1]).mean(dim=1)

    def encode(self, frames):
        with torch.no_grad():
            pv, grid = self._inputs(frames)
            out = self.visual(pv, grid_thw=grid)
            feats = self._pool_per_frame(out, len(frames))
        return feats.detach().float().cpu().numpy().astype(np.float32)

enc = FrozenQwenVisionEncoder(MODEL, device)

In [ ]:
def load_manifest(root: Path):
    with open(root / "manifest.csv", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

manifest = load_manifest(DATA)
train_rows = [r for r in manifest if r["split"] == "train"]
dev_rows   = [r for r in manifest if r["split"] == "validation"]
print("train", len(train_rows), "dev", len(dev_rows))

In [ ]:
def extract_rows(rows, cap, label):
    sel = rows[:cap] if cap else rows
    OUT.mkdir(parents=True, exist_ok=True)
    done = skipped = failed = 0
    for i, r in enumerate(sel, 1):
        target = OUT / f"{r['sample_id']}.npy"
        if target.exists():
            skipped += 1
            continue
        try:
            frames = sample_frames(DATA / r["video"], FRAMES, IMAGE)
            feats  = enc.encode(frames)
            np.save(target, feats)
            done += 1
        except Exception as e:
            failed += 1
            print("  fail", r["sample_id"], type(e).__name__, e, flush=True)
        if i % 25 == 0 or i == len(sel):
            print(f"[{label}] {i}/{len(sel)} done={done} skipped={skipped} failed={failed}", flush=True)
    print(f"[{label}] final done={done} skipped={skipped} failed={failed}")

extract_rows(train_rows, LIMIT, "train")
extract_rows(dev_rows, LIMIT, "dev")

In [ ]:
# discriminative probe: mean-pooled features, linear per-gloss ROC-AUC (train->dev)
if RUN_PROBE:
    from sklearn.linear_model import LogisticRegression
    from sklearn.multiclass import OneVsRestClassifier
    from sklearn.metrics import roc_auc_score
    from collections import Counter

    def gloss_tokens(label): return [t for t in str(label).split("/") if t]
    def build_labels(rows, min_freq=2, max_labels=60):
        cnt = Counter()
        for r in rows: cnt.update(set(gloss_tokens(r["label"])))
        labels = sorted((t for t, c in cnt.items() if c >= min_freq),
                        key=lambda t: (-cnt[t], t))[:max_labels]
        return labels, {t: i for i, t in enumerate(labels)}
    def load_matrix(rows, labels, idx):
        X, Y = [], []
        for r in rows:
            p = OUT / f"{r['sample_id']}.npy"
            if not p.exists(): continue
            y = np.zeros(len(labels), dtype=np.float32)
            for t in set(gloss_tokens(r["label"])):
                j = idx.get(t)
                if j is not None: y[j] = 1.0
            X.append(np.load(p).mean(axis=0))
            Y.append(y)
        return np.stack(X), np.stack(Y)

    labels, idx = build_labels(train_rows)
    Xtr, Ytr = load_matrix(train_rows, labels, idx)
    Xde, Yde = load_matrix(dev_rows, labels, idx)
    print("train", Xtr.shape, "dev", Xde.shape)
    if Xtr.shape[0] > 10 and Xde.shape[0] > 5:
        clf = OneVsRestClassifier(LogisticRegression(max_iter=3000))
        clf.fit(Xtr, Ytr)
        aucs = []
        proba = clf.predict_proba(Xde)
        for j in range(len(labels)):
            yt = Yde[:, j].astype(int)
            if yt.sum() > 0 and (yt == 0).sum() > 0:
                aucs.append(roc_auc_score(yt, proba[:, j]))
        macro = float(np.mean(aucs)) if aucs else 0.0
        print("==== MACRO_AUC =", round(macro, 4), "(", len(aucs), "labels) ====")
        if macro > 0.75:
            print("GREEN: cleared ~0.75 ceiling - run FULL (set LIMIT=None, re-run top cells, re-run this)")
        else:
            print("RED: no real gain over landmark 0.69 - stop; do not spend the full extract")

In [ ]:
# zip features for download back to the host (only for a full run)
if not LIMIT:
    z = Path("/kaggle/working") / "ce-csl-qwenvl7b.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(OUT.rglob("*.npy")):
            zf.write(f, f.relative_to(OUT.parent))
    print("zipped", z, "size", round(z.stat().st_size / 1e9, 2), "GB")
else:
    print("small trial only - features kept in /kaggle/working")

print("\nDONE")